# PCA reduction of review embeddings

Loads the raw 384-dim sentence embeddings from `06_nlp_embeddings.ipynb`
(`features_embeddings.parquet`) and compresses them to **20 principal
components**, saved as `features_embeddings_PCA.parquet`, keyed by `wine_id`.

Why a separate, persisted artifact:
- 384 raw dims are wide/noisy for an XGBoost on ~113k rows; ~20 PCs keep the
  dominant semantic axes while cutting overfitting surface.
- **PCA is unsupervised** — it never sees `retail`, so fitting on all rows is
  leakage-free and fine to cache as a Gold feature module (same rule as the
  ordinal encoding in `04_feature_engineering`).
- Re-runs cheaply from the cached embeddings (no re-encoding).

In [9]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

EMBEDDINGS_PATH = r"..\..\features\features_embeddings.parquet"
PCA_PATH        = r"..\..\features\features_embeddings_PCA.parquet"
N_COMPONENTS    = 20

emb = pd.read_parquet(EMBEDDINGS_PATH)
emb_cols = [c for c in emb.columns if c.startswith("emb_")]
print(f"embeddings: {emb.shape}  ({len(emb_cols)} dims)")
assert emb["wine_id"].is_unique, "wine_id must be unique"

embeddings: (135192, 385)  (384 dims)


## Fit PCA

`sklearn` PCA mean-centres internally. Embedding dims share a comparable
scale, so no separate standardisation is applied. `random_state` fixes the
(sign-arbitrary) solver for reproducibility.

In [10]:
X = emb[emb_cols].to_numpy(dtype=np.float32)

pca = PCA(n_components=N_COMPONENTS, random_state=42)
Z = pca.fit_transform(X).astype(np.float32)

evr = pca.explained_variance_ratio_
cum = evr.cumsum()
print(f"{N_COMPONENTS} components capture {cum[-1]:.1%} of embedding variance\n")
print(pd.DataFrame({
    "component": [f"pca_{i:02d}" for i in range(N_COMPONENTS)],
    "explained_var": evr.round(4),
    "cumulative": cum.round(4),
}).to_string(index=False))

20 components capture 46.5% of embedding variance

component  explained_var  cumulative
   pca_00         0.1091      0.1091
   pca_01         0.0392      0.1482
   pca_02         0.0318      0.1800
   pca_03         0.0290      0.2090
   pca_04         0.0256      0.2347
   pca_05         0.0226      0.2573
   pca_06         0.0211      0.2783
   pca_07         0.0194      0.2978
   pca_08         0.0181      0.3159
   pca_09         0.0171      0.3330
   pca_10         0.0158      0.3488
   pca_11         0.0146      0.3635
   pca_12         0.0143      0.3778
   pca_13         0.0140      0.3917
   pca_14         0.0133      0.4051
   pca_15         0.0129      0.4180
   pca_16         0.0127      0.4306
   pca_17         0.0121      0.4427
   pca_18         0.0116      0.4544
   pca_19         0.0108      0.4651


## Explained-variance curve

How much total variance is retained as components accumulate — eyeball
whether 20 is a reasonable elbow or if more/fewer would do.

In [11]:
fig = px.line(
    x=list(range(1, N_COMPONENTS + 1)), y=cum,
    markers=True,
    labels={"x": "components", "y": "cumulative explained variance"},
    title=f"PCA cumulative explained variance (first {N_COMPONENTS} of {len(emb_cols)})",
)
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

## Assemble & save

`wine_id` + `pca_00…pca_19` (float32). Joins to `features_basic` /
`features_keywords` on `wine_id`.

In [12]:
pca_cols = [f"pca_{i:02d}" for i in range(N_COMPONENTS)]
out = pd.concat([
    emb[["wine_id"]].reset_index(drop=True),
    pd.DataFrame(Z, columns=pca_cols),
], axis=1)

out.to_parquet(PCA_PATH, index=False)
size_mb = __import__("os").path.getsize(PCA_PATH) / 1e6
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols ({size_mb:.1f} MB) -> {PCA_PATH}")
out.head()

Saved 135,192 rows x 21 cols (17.2 MB) -> ..\..\.data\features_embeddings_PCA.parquet


,wine_id,pca_00,pca_01,pca_02,pca_03,pca_04,pca_05,pca_06,pca_07,pca_08,...,pca_10,pca_11,pca_12,pca_13,pca_14,pca_15,pca_16,pca_17,pca_18,pca_19
0,0,0.087705,-0.032049,-0.131868,0.126768,-0.069865,-0.018574,0.082918,-0.246926,-0.015410,...,0.016126,0.108556,-0.009206,-0.134355,-0.038856,0.000008,-0.080921,-0.094215,0.023154,0.064461
1,1,-0.248415,-0.150930,0.114767,0.100128,-0.002249,0.114769,-0.213825,-0.011266,-0.072281,...,0.065217,0.056112,0.011863,0.000809,-0.034462,0.007005,0.079466,-0.100018,0.131615,-0.016012
2,2,-0.266211,-0.170863,0.059909,-0.050827,-0.066372,0.031644,0.044207,0.067166,0.028612,...,0.041501,0.039497,0.131129,-0.015150,0.043448,0.072358,-0.047951,-0.024668,0.025297,-0.119325
3,3,0.331886,-0.048889,0.056259,0.163971,-0.037124,0.032894,-0.031904,0.002716,0.003294,...,0.087743,0.025998,-0.029764,-0.147797,0.136203,0.018809,-0.019578,0.124389,0.066332,0.006394
4,4,0.226620,0.044751,0.122244,-0.025042,0.025569,0.049791,-0.102695,-0.019718,0.064431,...,-0.056579,0.106334,-0.039858,0.027398,0.064483,-0.132130,0.009157,-0.098717,0.030304,-0.042359


## Next step

`models/01_models.ipynb` loads this as **model 3** (`features_basic` +
`pca_*`) and compares R² against the current baseline (model 1). If the PCs
help, follow up by tuning `N_COMPONENTS` (e.g. 32 / 50) and re-running this
notebook — the model notebook picks up whatever this writes.